# Configurações iniciais com banco de dados

### Configurando o jdk

In [121]:
import os
import subprocess
from pathlib import Path

import jdk

JDK_VERSION = "17"
JDK_DIR = Path.cwd().parent.parent.parent / ".jdk"


def get_java_home(base: Path):
    """Procura a pasta que contém bin/java dentro de base."""
    exe = "java.exe" if os.name == "nt" else "java"
    for caminho in base.rglob(f"bin/{exe}"):
        return caminho.parent.parent
    return None


java_home = get_java_home(JDK_DIR) if JDK_DIR.exists() else None

if java_home is None:
    print("Baixando o JDK...")
    jdk.install(JDK_VERSION, path=str(JDK_DIR))
    java_home = get_java_home(JDK_DIR)

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ["PATH"]

print("JAVA_HOME =", java_home)
subprocess.run(["java", "-version"])

JAVA_HOME = /home/mateus/Code/Eng-Dados/apache-spark-minio/.jdk/jdk-17.0.20.1+1


openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment Temurin-17.0.20.1+1 (build 17.0.20.1+1)
OpenJDK 64-Bit Server VM Temurin-17.0.20.1+1 (build 17.0.20.1+1, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

### Configurando o banco de dados

In [122]:
from dotenv import load_dotenv

load_dotenv(override=True)

DB_HOST = os.getenv("HOST")
DB_PORT = os.getenv("PORT")
DB_NAME = os.getenv("DATABASE")
DB_USER = os.getenv("USER")
DB_PASS = os.getenv("PASSWORD")

print(f"Server: {DB_HOST}:{DB_PORT}")
print(f"Database: {DB_NAME}")

Server: 127.0.0.1:5432
Database: delta_games


### Montando a string de conexão

In [123]:
import pyodbc

conn_str = (
    "DRIVER={PostgreSQL Unicode};"
    f"SERVER={DB_HOST};"
    f"PORT={DB_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={DB_USER};"
    f"PWD={DB_PASS};"
)

### Importando o arquivo de DDL.sql

In [124]:
sql_path = Path("../database/DDL.sql")

sql = sql_path.read_text(encoding="utf-8")

with pyodbc.connect(conn_str) as conn:
    cursor = conn.cursor()
    cursor.execute(sql)

print("Tabelas do branco criadas")

Tabelas do branco criadas


## Adicionando os arquivos CSV para o Banco de dados

#### Configurando pyspark

In [125]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("csv_para_banco")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

#### Lendo o arquivo CSV

In [126]:
csv_path = JDK_DIR.parent / "data" / "metacritic_games_not_null.csv"

df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

In [127]:
url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"

props = {
    "user": DB_USER,
    "password": DB_PASS,
    "driver": "org.postgresql.Driver"
}

df.write.jdbc(url=url, table="games", mode="append", properties=props)

#### Conferindo os dados gravados

In [128]:
spark.read.jdbc(url, "games", properties=props).count()

21910

In [129]:
from pyspark.sql.functions import col

sv_df = spark.read.jdbc(url, "games", properties=props)
sv_df = sv_df.where(col("platform") == "PC")
sv_df.count()

6282

In [130]:
steam_df = spark.read.csv(
    str(csv_path.parent / "steam_metrics.csv"),
    header=True,
    inferSchema=True
)

steam_df = steam_df.withColumnsRenamed({
    "AppID": "steam_appid",
    "Name": "name",
    "Price": "price",
    "Achievements": "achievements",
    "Positive": "positive_reviews",
    "Negative": "negative_reviews",
    "Average playtime forever": "avg_playtime"
})

gv_df = steam_df.join(sv_df.select("name", "id"), on="name", how="inner")
gv_df = gv_df.withColumnRenamed("id", "game_id")
gv_df.show(5)

+--------------------+-----------+-----+----------------+----------------+------------+------------+-------+
|                name|steam_appid|price|positive_reviews|negative_reviews|achievements|avg_playtime|game_id|
+--------------------+-----------+-----+----------------+----------------+------------+------------+-------+
|               #BLUD|    1921480|14.99|             251|              65|          43|         343|      1|
|     #KILLALLZOMBIES|     303720| 4.99|             242|              67|          10|         329|      4|
|.hack//G.U. Last ...|     525480| 9.99|            3358|             514|          52|        1040|      9|
|           0RBITALIS|     278440| 9.99|             219|              88|          38|         179|     34|
|     10 Second Ninja|     271670| 1.99|             659|             115|          30|         104|     36|
+--------------------+-----------+-----+----------------+----------------+------------+------------+-------+
only showing top 5 

In [131]:
gv_df = gv_df.drop("name")

gv_df.write.jdbc(url=url, table="steam_metrics", mode="append", properties=props)

In [132]:
spark.read.jdbc(url, "steam_metrics", properties=props).count()

3178

#### Gerando um DataFrame padrão e limpando os existentes

In [133]:
std_df = spark.read.jdbc(url, "games", properties=props).select(col("id").alias("game_id"), col("name"))
std_df.show()

del df
del steam_df
del gv_df
del sv_df

+-------+--------------------+
|game_id|                name|
+-------+--------------------+
|      1|               #BLUD|
|      2|              #DRIVE|
|      3|              #IDARB|
|      4|     #KILLALLZOMBIES|
|      5|     #KILLALLZOMBIES|
|      6|       'Splosion Man|
|      7|            .detuned|
|      8|.hack//G.U. Last ...|
|      9|.hack//G.U. Last ...|
|     10|.hack//G.U. Last ...|
|     11|.hack//G.U. vol. ...|
|     12|.hack//G.U. vol. ...|
|     13|.hack//G.U. vol. ...|
|     14|.hack//Infection ...|
|     15|.hack//Mutation P...|
|     16|.hack//Outbreak P...|
|     17|0 Day Attack on E...|
|     18|         007 Legends|
|     19|         007 Legends|
|     20|         007 Legends|
+-------+--------------------+
only showing top 20 rows


#### Lendo e gravando as vendas dos jogos

In [134]:
sales = spark.read.csv(
    str(csv_path.parent / "sales.csv"),
    header=True,
    inferSchema=True
)

sales = sales.withColumnsRenamed({
    "Name": "name",
    "Platform": "platform",
    "NA_Sales": "na_sales",
    "EU_Sales": "eu_sales",
    "JP_Sales": "jp_sales",
    "Other_Sales": "other_sales",
    "Global_Sales": "global_sales"
})

sales = sales.join(std_df.select("name", "game_id"), on="name", how="inner")

sales.show(5)

+--------------------+--------+--------+--------+--------+-----------+------------+-------+
|                name|platform|na_sales|eu_sales|jp_sales|other_sales|global_sales|game_id|
+--------------------+--------+--------+--------+--------+-----------+------------+-------+
|.hack//Infection ...|     PS2|    0.49|    0.38|    0.26|       0.13|        1.27|     14|
|.hack//Mutation P...|     PS2|    0.23|    0.18|     0.2|       0.06|        0.68|     15|
|.hack//Outbreak P...|     PS2|    0.14|    0.11|    0.17|       0.04|        0.46|     16|
|          007 Racing|      PS|     0.3|     0.2|     0.0|       0.03|        0.53|     21|
|007: Quantum of S...|      PC|    0.01|    0.01|     0.0|        0.0|        0.03|     27|
+--------------------+--------+--------+--------+--------+-----------+------------+-------+
only showing top 5 rows


In [135]:
sales = sales.drop("name", "platform")

sales.write.jdbc(url, "regional_sales", mode="append", properties=props)

#### Verificando se foi gravado com sucesso

In [136]:
spark.read.jdbc(url, "regional_sales", properties=props).count()

del sales

#### Lendo o CSV e gravando em HowLongToBeat

In [137]:
hltb_df = spark.read.csv(
    str(csv_path.parent / "hltb_corr_values.csv"),
    header=True,
    inferSchema=True
)

hltb_df.show(5)

+---+--------------------+----+--------------------+----------+---------------+-------------+
| id|                name|type|            platform|main_story|main_plus_sides|completionist|
+---+--------------------+----+--------------------+----------+---------------+-------------+
|  1|       'Splosion Man|game|            Xbox 360|       7.8|           9.47|        17.62|
| 10|         007 Legends|game|PC, PlayStation 3...|      6.46|           8.21|         9.69|
| 19|101-in-1 Explosiv...|game|    Nintendo DS, Wii|      1.35|           2.67|        14.14|
| 24|18 Wheeler: Ameri...|game|Dreamcast, Ninten...|      0.86|           1.65|         1.66|
| 35|   3 on 3 NHL Arcade|game|PlayStation 3, Xb...|      1.12|           1.99|         2.86|
+---+--------------------+----+--------------------+----------+---------------+-------------+
only showing top 5 rows


In [138]:
hltb_df = hltb_df.join(std_df.select("name", "game_id"), on="name", how="inner")
hltb_df = hltb_df.withColumnsRenamed({ "main_story": "main_story_hours", "main_plus_sides": "extras_hours", "completionist": "completionist_hours" })
hltb_df.show(5)

+-------------+-----+----+--------------------+----------------+------------+-------------------+-------+
|         name|   id|type|            platform|main_story_hours|extras_hours|completionist_hours|game_id|
+-------------+-----+----+--------------------+----------------+------------+-------------------+-------+
|        #BLUD|45648|game|Nintendo Switch, ...|            9.23|       11.09|              17.91|      1|
|'Splosion Man|    1|game|            Xbox 360|             7.8|        9.47|              17.62|      6|
|     .detuned| 9627|game|       PlayStation 3|            0.17|        0.33|               0.41|      7|
|  007 Legends|   10|game|PC, PlayStation 3...|            6.46|        8.21|               9.69|     18|
|  007 Legends|   10|game|PC, PlayStation 3...|            6.46|        8.21|               9.69|     19|
+-------------+-----+----+--------------------+----------------+------------+-------------------+-------+
only showing top 5 rows


In [139]:
hltb_df = hltb_df.drop("name", "id", "type", "platform")

hltb_df.write.jdbc(url, "how_long_to_beat", mode="append", properties=props)

In [140]:
spark.read.jdbc(url, "how_long_to_beat", properties=props).count()

13196